# Module 4: Imbalanced Data and Threshold Selection
**ACS Predictive Analytics Curriculum**

Concepts covered:
- Why accuracy fails with imbalanced data
- SMOTE vs alternatives in R (themis package)
- Precision-recall curves
- F2 score for child welfare (recall-weighted)
- Threshold selection as an operational decision
- Fairness audit by borough


In [ ]:
library(tidyverse)
library(tidymodels)
library(themis)
library(probably)
library(yardstick)

scr      <- read_csv('data/acs_scr_reports.csv', show_col_types=FALSE)
features <- read_csv('data/acs_features.csv',    show_col_types=FALSE)
cat('Data loaded\n')

## SECTION 4.1: The Problem With Accuracy

In [ ]:
# Build minimal clean dataset
df <- features %>%
  mutate(
    target = factor(needs_investigative_consultation,
                    levels=c(0,1), labels=c('No','Yes'))
  )

# Class distribution
df %>% count(target) %>%
  mutate(pct = round(n/sum(n)*100,1))

# The naive classifier: always predict 'No'
naive_accuracy <- mean(df$target == 'No')
cat('Naive accuracy (always predict No):', round(naive_accuracy*100,1), '%\n')
cat('But recall for high-risk cases:', 0, '%\n')
cat('Every high-risk child is missed!\n')

# DOMAIN INSIGHT:
# A model predicting No for every case would have ~75% accuracy
# but would be completely useless for child welfare
# This is why we use AUC-PR, not accuracy

## SECTION 4.2: SMOTE in R with themis

In [ ]:
# SMOTE must be applied INSIDE the recipe
# Never before train/test split

# Show effect of SMOTE on class distribution
pre_smote <- df %>% count(target)
cat('Before SMOTE:\n')
print(pre_smote)

# Apply SMOTE via recipe
smote_recipe <- recipe(target ~ prior_reports_12mo + dv_history_flag +
                        child_age_under_5 + shelter_involvement_flag,
                        data=df) %>%
  step_smote(target, over_ratio=1.0, seed=42)

prepped <- prep(smote_recipe)
baked   <- bake(prepped, new_data=NULL)

post_smote <- baked %>% count(target)
cat('\nAfter SMOTE (balanced):\n')
print(post_smote)

cat('\nNote: SMOTE creates SYNTHETIC cases by interpolating')
cat('\nbetween existing minority class samples')
cat('\nNever apply to test data - training fold only!\n')

## SECTION 4.3: Precision-Recall Curve

In [ ]:
library(ranger)
set.seed(42)

# Quick model for demonstration
train_idx <- sample(nrow(df), 0.8*nrow(df))
train <- df[train_idx,]
test  <- df[-train_idx,]

# Features for quick demo
FEATS <- c('prior_reports_12mo','prior_substantiated_flag',
           'dv_history_flag','child_age_under_5',
           'shelter_involvement_flag','reporter_accuracy_score')

# Train RF
rf <- rand_forest(trees=100) %>%
  set_engine('ranger') %>%
  set_mode('classification') %>%
  fit(target ~ ., data=train %>% select(all_of(c(FEATS,'target'))))

# Get probabilities
preds <- predict(rf, test, type='prob') %>%
  bind_cols(test %>% select(target))

# Precision-recall curve
pr_curve_data <- preds %>%
  pr_curve(truth=target, .pred_Yes)

pr_curve_data %>%
  ggplot(aes(x=recall, y=precision)) +
  geom_path(color='#CB181D', linewidth=1.2) +
  geom_point(data=pr_curve_data %>% filter(abs(recall-0.70)<0.02) %>% slice(1),
             size=4, color='navy') +
  annotate('text', x=0.72, y=0.45, label='Threshold=0.40\n(recommended)',
           hjust=0, color='navy', size=4) +
  geom_hline(yintercept=mean(test$target=='Yes'), linetype='dashed',
             color='gray50', linewidth=0.8) +
  annotate('text', x=0.05, y=mean(test$target=='Yes')+0.03,
           label='Baseline (random)', color='gray50', size=3.5) +
  labs(
    title    = 'Precision-Recall Curve — ACS Risk Model',
    subtitle = 'Every point = one possible threshold | Dashed = random baseline',
    x        = 'Recall (% of true high-risk cases caught)',
    y        = 'Precision (% of flags that are correct)'
  ) +
  theme_minimal(base_size=12) +
  theme(plot.title=element_text(face='bold'))

## SECTION 4.4: Threshold Selection with F2 Score

In [ ]:
# F2 score weights recall TWICE as heavily as precision
# Right choice for child welfare: missing a case costs more than false alarm

# Evaluate at multiple thresholds
thresholds <- seq(0.25, 0.65, by=0.05)

threshold_results <- map_dfr(thresholds, function(thresh) {
  preds_thresh <- preds %>%
    mutate(
      .pred_class = factor(ifelse(.pred_Yes >= thresh, 'Yes', 'No'),
                           levels=c('No','Yes'))
    )

  cm  <- conf_mat(preds_thresh, truth=target, estimate=.pred_class)$table
  tp  <- cm['Yes','Yes']
  fp  <- cm['No','Yes']
  fn  <- cm['Yes','No']
  tn  <- cm['No','No']

  recall_v    <- tp / (tp + fn)
  precision_v <- tp / (tp + fp)
  f2          <- ifelse(precision_v + recall_v > 0,
                   (1 + 4) * precision_v * recall_v / (4*precision_v + recall_v), 0)
  # Daily flags: scale to ~140 ACS reports/day citywide
  daily_flags <- round((tp + fp) / nrow(test) * 140)

  tibble(threshold=thresh, recall=round(recall_v,3), precision=round(precision_v,3),
         f2_score=round(f2,3), daily_flags=daily_flags)
})

print(threshold_results)

# Visualize
threshold_results %>%
  pivot_longer(c(recall, precision, f2_score), names_to='metric', values_to='value') %>%
  ggplot(aes(x=threshold, y=value, color=metric)) +
  geom_line(linewidth=1) +
  geom_vline(xintercept=threshold_results$threshold[which.max(threshold_results$f2_score)],
             linetype='dashed', color='black') +
  annotate('text', x=threshold_results$threshold[which.max(threshold_results$f2_score)]+0.01,
           y=0.9, label='F2 optimal', hjust=0) +
  scale_color_manual(values=c(recall='#CB181D', precision='#2171B5', f2_score='#238B45')) +
  labs(title='Threshold Selection: Precision vs Recall vs F2',
       subtitle='F2 weights recall 2x — right choice for child welfare',
       x='Classification Threshold', y='Score', color='Metric') +
  theme_minimal(base_size=12)

## SECTION 4.5: Fairness Audit by Borough

In [ ]:
# FATML PRINCIPLE: Never report only aggregate metrics
# A model performing well overall but poorly in Bronx is NOT fair

borough_audit <- preds %>%
  bind_cols(test %>% select(family_id)) %>%
  left_join(scr %>% select(family_id, borough) %>% distinct(),
            by='family_id') %>%
  mutate(.pred_class = factor(ifelse(.pred_Yes >= 0.40, 'Yes', 'No'),
                              levels=c('No','Yes'))) %>%
  group_by(borough) %>%
  summarise(
    n              = n(),
    positive_rate  = round(mean(target=='Yes')*100, 1),
    model_flag_rate= round(mean(.pred_class=='Yes')*100, 1),
    avg_score      = round(mean(.pred_Yes), 3),
    recall         = {
      tp <- sum(target=='Yes' & .pred_class=='Yes')
      fn <- sum(target=='Yes' & .pred_class=='No')
      round(tp/(tp+fn), 3)
    },
    false_pos_rate = {
      fp <- sum(target=='No' & .pred_class=='Yes')
      tn <- sum(target=='No' & .pred_class=='No')
      round(fp/(fp+tn), 3)
    },
    .groups='drop'
  ) %>%
  filter(!is.na(borough)) %>%
  arrange(desc(false_pos_rate))

print(borough_audit)

# INTERPRETATION:
# If false_pos_rate is higher in Bronx/Brooklyn than Staten Island
# the model generates more unnecessary investigation flags
# in majority Black/Latino boroughs -> equity concern
# Action: report to ACS equity team before deployment